In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Path to dataset files: /kaggle/input/skin-cancer9-classesisic


In [3]:
# Cell 1: Install Dependencies
!pip install -q timm thop scikit-learn xgboost torchinfo

import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import timm
from thop import profile
from torchinfo import summary
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from xgboost import XGBClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [11]:
# Cell 2: Load Real ISIC Dataset from kagglehub Path
import os
import kagglehub
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# Download dataset
path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")
print("Path to dataset files:", path)

# Define standard image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# The main dataset directory that contains 'Test' and 'Train' folders
base_dataset_path = os.path.join(path, "Skin cancer ISIC The International Skin Imaging Collaboration")

# Load Train and Test datasets separately
train_dataset = ImageFolder(root=os.path.join(base_dataset_path, "Train"), transform=transform)
test_dataset = ImageFolder(root=os.path.join(base_dataset_path, "Test"), transform=transform)

print(f"Total images found in Train: {len(train_dataset)}")
print(f"Classes in Train: {train_dataset.classes}")
print(f"Total images found in Test: {len(test_dataset)}")
print(f"Classes in Test: {test_dataset.classes}")

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)


Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Path to dataset files: /kaggle/input/skin-cancer9-classesisic
Total images found in Train: 2239
Classes in Train: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']
Total images found in Test: 118
Classes in Test: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']


In [7]:
# Cell 3: Transfer Learning Training & Evaluation Loop (Tables 1 & 3)
model_names = ['vgg16', 'resnet18', 'efficientnet_b0'] # Removed 'alexnet' as it was not recognized
results_table1 = []
results_table3 = []

for name in model_names:
    print(f"\n--- Training & Benchmarking: {name} ---")
    # Load model with timm, setting num_classes to 9
    model = timm.create_model(name, pretrained=True, num_classes=9).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)

    # 1. Compute Efficiency Metrics (Table 3 inputs)
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    try:
        macs, params = profile(model, inputs=(dummy_input,), verbose=False)
        flops_g = (macs * 2) / 1e9 # Convert MACs to FLOPs
    except:
        flops_g, params = 0.0, sum(p.numel() for p in model.parameters())

    param_m = params / 1e6

    # Model Size in MB
    torch.save(model.state_dict(), "temp.pth")
    import os
    model_size_mb = os.path.getsize("temp.pth") / (1024 * 1024)
    os.remove("temp.pth")

    # Quick 1-epoch training simulation
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        break # Just 1 batch for demonstration script speed

    # Evaluation & Inference Time (Table 1 & 3)
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    start_time = time.time()
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    total_time = time.time() - start_time
    inference_time_ms = (total_time / len(test_dataset)) * 1000

    acc = accuracy_score(all_labels, all_preds) * 100
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)

    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro') * 100
    except:
        auc = 0.0

    results_table1.append({
        "Model": name, "Accuracy (%)": f"{acc:.2f}", "Precision (%)": f"{precision*100:.2f}",
        "Recall (%)": f"{recall*100:.2f}", "F1-Score (%)": f"{f1*100:.2f}", "AUC (%)": f"{auc:.2f}"
    })

    results_table3.append({
        "Model": name, "Parameters (M)": f"{param_m:.2f}", "Model Size (MB)": f"{model_size_mb:.2f}",
        "FLOPs (G)": f"{flops_g:.2f}", "Inference Time (ms)": f"{inference_time_ms:.2f}", "Accuracy (%)": f"{acc:.2f}"
    })

print("\nPipeline Complete!")


--- Training & Benchmarking: vgg16 ---


model.safetensors: reconstructing file:   0%|          |  0.00B /  553MB            

model.safetensors: downloading bytes:           |  0.00B            


--- Training & Benchmarking: resnet18 ---


model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            


--- Training & Benchmarking: efficientnet_b0 ---


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            


Pipeline Complete!


In [12]:
# Cell 4: Deep Feature Extraction & Classical Classifiers (Table 2)
# Using a feature extractor backbone (e.g., ResNet18 without final classification head)
backbone = timm.create_model('resnet18', pretrained=True, num_classes=0).to(device)
backbone.eval()

def extract_features(loader):
    features, targets = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            feats = backbone(images)
            features.append(feats.cpu().numpy())
            targets.append(labels.numpy())
    return np.vstack(features), np.concatenate(targets)

X_train, y_train = extract_features(train_loader)
X_test, y_test = extract_features(test_loader)

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(),
    "Linear SVM": LinearSVC(max_iter=1000),
    "RBF-SVM": SVC(probability=True),
    "XGBoost": XGBClassifier()
}

table2_results = []
for clf_name, clf in classifiers.items():
    print(f"Training Classifier: {clf_name}...")

    # Check if there's more than one class in y_train before training
    if len(np.unique(y_train)) < 2:
        print(f"Skipping {clf_name}: Only one class found in training data (y_train). Please ensure the dataset loading in Cell 2 correctly identifies multiple classes.")
        table2_results.append({
            "Classifier": clf_name, "Accuracy (%)": "N/A",
            "Precision (%)": "N/A", "Recall (%)": "N/A",
            "F1-Score (%)": "N/A"
        })
        continue # Skip to the next classifier

    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    acc = accuracy_score(y_test, preds) * 100
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average='macro', zero_division=0)

    table2_results.append({
        "Classifier": clf_name, "Accuracy (%)": f"{acc:.2f}",
        "Precision (%)": f"{prec*100:.2f}", "Recall (%)": f"{rec*100:.2f}",
        "F1-Score (%)": f"{f1*100:.2f}"
    })


Training Classifier: Logistic Regression...
Training Classifier: Decision Tree...
Training Classifier: Random Forest...
Training Classifier: K-Nearest Neighbors (KNN)...
Training Classifier: Linear SVM...
Training Classifier: RBF-SVM...
Training Classifier: XGBoost...


### Table 1: Transfer Learning Model Performance

In [15]:
import pandas as pd
df_table1 = pd.DataFrame(results_table1)
display(df_table1)

,Model,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
0,vgg16,100.00,100.00,100.00,100.00,0.00
1,resnet18,2.54,12.50,0.32,0.62,0.00
2,efficientnet_b0,18.01,12.50,2.25,3.82,0.00


### Table 2: Deep Feature Extraction with Classical Classifiers Performance

In [16]:
df_table2 = pd.DataFrame(table2_results)
display(df_table2)

,Classifier,Accuracy (%),Precision (%),Recall (%),F1-Score (%)
0,Logistic Regression,50.00,47.74,50.00,44.43
1,Decision Tree,25.42,18.55,26.85,20.17
2,Random Forest,36.44,43.75,35.88,30.82
3,K-Nearest Neighbors (KNN),28.81,33.03,29.63,28.32
4,Linear SVM,45.76,45.09,46.53,42.63
5,RBF-SVM,49.15,52.82,49.31,44.43
6,XGBoost,38.98,36.33,40.97,34.60


### Table 3: Transfer Learning Model Efficiency and Performance

In [17]:
df_table3 = pd.DataFrame(results_table3)
display(df_table3)

,Model,Parameters (M),Model Size (MB),FLOPs (G),Inference Time (ms),Accuracy (%)
0,vgg16,134.30,512.32,30.93,650.10,100.00
1,resnet18,11.18,42.74,3.65,89.51,2.54
2,efficientnet_b0,3.98,15.73,0.77,82.32,18.01
